In [1]:
# LandownerElkTags
import tabula
import pandas as pd
import requests
import io
import numpy as np
from publictrust.wildlife import parse_pdf_to_dataframe, create_interactive_map
import geopandas as gpd
from pathlib import Path



In [2]:

landownertags2025_url = "https://wgfd.wyo.gov/media/32688/download?inline" # 2025 Landowner Elk Tag Quotas PDF

# Get the DataFrame
landowner_tags = parse_pdf_to_dataframe(landownertags2025_url)

if not landowner_tags.empty:
    # Determine percentage of tags allocated to the landowner
    landowner_tags['pct_landowner'] = landowner_tags['Issued']/landowner_tags['Quota'] * 100
    ls_tag_dist = []
    for i, row in landowner_tags.iterrows():
        ls_tag_dist.append(f"{row['Issued']} of {row['Quota']}")
    landowner_tags['landowner_tags_per_total'] = ls_tag_dist
else:
    print("PROBLEM WITH READING DATA FROM THE PDF FILE. CANNOT PROCEED!")

Successfully parsed PDF and created DataFrame.


In [10]:
# Review summaries of landowner & resident quota data used in the maps:
print(landowner_tags)

   Hunt_Area Type     Description  Quota  Issued  PP  Applicants  \
0          1    1         ANY ELK     84       2   1           0   
1          1    4  ANTLERLESS ELK     63       0   0           0   
2          6    1         ANY ELK     42       0   0           0   
3          6    4  ANTLERLESS ELK     21       0   0           0   
4          7    1         ANY ELK   1260     253   0           0   
..       ...  ...             ...    ...     ...  ..         ...   
25       123    4  ANTLERLESS ELK     90       0   0           0   
26       124    1         ANY ELK     84      24   0           0   
27       124    4  ANTLERLESS ELK    126       0   0           0   
28       125    1         ANY ELK    168       3   0           0   
29       GEN  NaN         GENERAL   9999       0   0           0   

    pct_landowner landowner_tags_per_total  
0        2.380952                  2 of 84  
1        0.000000                  0 of 63  
2        0.000000                  0 of 42  
3  

In [ ]:
# Read in the Wyoming G&F Elk Hunt Areas Map
gdf_ha = gpd.read_file("~/Documents/personal/BHA/data_in/gpkg/ElkHuntAreas_5201991498639818979.gpkg")
gdf_ha['HUNTAREA'] = gdf_ha['HUNTAREA'].astype(str).str.replace(".0","")


,HUNTAREA,HERDUNIT,HERDNAME,SqMiles,HUNTNAME,Region,geometry
0,38,321.0,North Bighorn,564.013605,Tongue,Western,"MULTIPOLYGON (((-12001726.117 5621543.668, -12..."
1,39,321.0,North Bighorn,230.103979,Deer Creek,Western,"MULTIPOLYGON (((-12043953.543 5621502.72, -120..."
2,37,321.0,North Bighorn,683.693631,Goose,Western,"MULTIPOLYGON (((-11921824.71 5608346.64, -1192..."
3,40,321.0,North Bighorn,462.427588,Horse Creek,Western,"MULTIPOLYGON (((-12040582.169 5597205.616, -12..."
4,55,216.0,Cody,392.133906,Grinnell,Western,"MULTIPOLYGON (((-12227736.194 5567279.333, -12..."


In [8]:
# Combine map data with the landowner tags data
gdf_ha_cmbo = gdf_ha.merge(landowner_tags,left_on='HUNTAREA', right_on='Hunt_Area')
gdf_cmbo_type1 = gdf_ha_cmbo[gdf_ha_cmbo['Type']=='1'] # Filter to the type 1 tags

In [5]:
create_interactive_map(gdf=gdf_cmbo_type1, output_html = "~/Documents/personal/BHA/data_out/updated_elk2025_landowner_type1_bha.html", map_title = '2025 Type 1 Resident Landowner Elk Tag Allocations',img_path = "~/Documents/personal/BHA/data_in/BHALOGOBLACK.clear.png")

Reprojecting data to WGS84 (EPSG:4326)...
Adding polygons and hover interactions...


/Users/guylitt/git/public_trust_wildlife/publictrust/wildlife.py:128: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center_lat = gdf.geometry.centroid.y.mean()
/Users/guylitt/git/public_trust_wildlife/publictrust/wildlife.py:129: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center_lon = gdf.geometry.centroid.x.mean()


Adding labels...
Map successfully generated: /Users/guylitt/Documents/personal/BHA/data_out/updated_elk2025_landowner_type1_bha.html
